# Attention Visualization

## Implementation

In [1]:
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from generate import load_model_from_checkpoint, encode_prompt


def extract_attention_matrices(model, input_ids, device):
    input_tensor = torch.tensor([input_ids], dtype=torch.long, device=device)
    with torch.no_grad():
        _, _, attention_weights_per_layer = model(input_tensor)
    return attention_weights_per_layer


def visualize_head_attention(attention_matrix, output_path, title):
    matrix = attention_matrix.detach().cpu().numpy()
    plt.figure(figsize=(8, 6))
    plt.imshow(matrix, cmap="viridis")
    plt.colorbar(label="Attention Weight")
    plt.xlabel("Key Position")
    plt.ylabel("Query Position")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(output_path)
    plt.close()

## Demonstration

In [2]:
torch.manual_seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint_path = "best_model.pt"

model, string_to_index, index_to_string, context_length = load_model_from_checkpoint(checkpoint_path, device)

prompt = "Shall I compare thee to a summer's day"
encoded_prompt = encode_prompt(prompt, string_to_index)
encoded_prompt = encoded_prompt[:context_length]

attention_weights_per_layer = extract_attention_matrices(model, encoded_prompt, device)

print("Attention Visualization")
print(f"Prompt: {prompt}")
print(f"Sequence Length: {len(encoded_prompt)}")
print(f"Number Of Layers: {len(attention_weights_per_layer)}")

last_layer_attention = attention_weights_per_layer[-1]
num_heads = last_layer_attention.shape[1]
print(f"Number Of Heads In Last Layer: {num_heads}")

for head_index in range(num_heads):
    head_attention_matrix = last_layer_attention[0, head_index]
    output_path = f"attention_head_{head_index + 1}.png"
    title = f"Last Layer Attention Head {head_index + 1}"
    visualize_head_attention(head_attention_matrix, output_path, title)
    print(f"Saved {output_path}")
    print(f"Head {head_index + 1} Attention Weights Shape: {tuple(head_attention_matrix.shape)}")
    print(f"Head {head_index + 1} Row Sum Check: {head_attention_matrix[0].sum().item():.6f}")

Attention Visualization
Prompt: Shall I compare thee to a summer's day
Sequence Length: 38
Number Of Layers: 4
Number Of Heads In Last Layer: 4
Saved attention_head_1.png
Head 1 Attention Weights Shape: (38, 38)
Head 1 Row Sum Check: 1.000000


Saved attention_head_2.png
Head 2 Attention Weights Shape: (38, 38)
Head 2 Row Sum Check: 1.000000
Saved attention_head_3.png
Head 3 Attention Weights Shape: (38, 38)
Head 3 Row Sum Check: 1.000000


Saved attention_head_4.png
Head 4 Attention Weights Shape: (38, 38)
Head 4 Row Sum Check: 1.000000
